# MATH840 — Lab 4: Forecasting with decomposition

**Week 4 | graded assignment 3 of 7 | 8 points | due 23:59 today**

Today you are issued **your own time series**, and it stays yours for the rest of the course. This lab
takes it apart and forecasts it from its parts: decompose, forecast the seasonally adjusted series,
carry the season forward, put them back — then compare the result honestly against the four simple
methods from Week 1. Toolbox only: no ETS, no ARIMA.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | EDA and visualisation | 1 |
| 2 | Decomposition and forecast | 3 |
| 3 | Justification block | 3 |
| 4 | Code quality and reproducibility | 1 |

**Accuracy is measured but not marked this week.** Your score against all four simple methods is
published tomorrow morning as information — it is the baseline the Week 6 challenge will be compared
against. The marks are for how the decomposition is done and how well you argue from it.

Submit `SURNAME_lab04_forecast.csv`, `SURNAME_lab04.pdf` and `SURNAME_lab04.ipynb` to the Week 4
activity on Moodle before 23:59.

## 0. Setup

Given — run it and move on. Nothing in this section is marked.

In [ ]:
!pip install -q statsforecast utilsforecast coreforecast

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from coreforecast.scalers import boxcox, boxcox_lambda

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you have used since Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

### Your series

The same identifier always produces the same series, and this series stays yours for Weeks 6, 7, 8
and 9. It is real data with its identity removed: no name, no units, values rescaled, the calendar
shifted by whole years, the history trimmed.

Every series in the pool is seasonal — there is something to decompose — and has at least fifteen
years of history.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data/trackb")


def assign_series(student_id: str, codes: list[str]) -> str:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


index = pd.read_csv(f"{BASE}/index.csv")
CODE = assign_series(MY_ID, sorted(index["code"]))
spec = index.set_index("code").loc[CODE]

M    = int(spec["m"])       # seasonal period: 12 monthly, 4 quarterly
H    = int(spec["h"])       # how many steps you are forecasting
FREQ = spec["freq"]         # pandas frequency of the calendar: "MS" or "QS"

s = pd.read_csv(f"{BASE}/{CODE}.csv", parse_dates=["ds"])
y = s["y"].to_numpy(float)

print(f"series {CODE}: {len(s)} observations, {s['ds'].min().date()} to {s['ds'].max().date()}")
print(f"seasonal period m = {M}, forecast horizon h = {H}, frequency {FREQ}")
s.tail(3)

### Helpers

Given, and used in exactly this form when your file is read. `mase` is scaled by the in-sample
seasonal naive error of **your whole history**, so the number you compute on your validation window is
on the same scale as the published one.

In [ ]:
def benchmark_forecasts(train: np.ndarray, h: int, m: int) -> dict[str, np.ndarray]:
    """The four simple methods from Week 1."""
    T = len(train)
    return {
        "mean":   np.repeat(train.mean(), h),
        "naive":  np.repeat(train[-1], h),
        "snaive": np.array([train[-m + (i % m)] for i in range(h)]),
        "drift":  train[-1] + np.arange(1, h + 1) * (train[-1] - train[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))      # MASE denominator, fixed for this series


def mase(actual, forecast) -> float:
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


FUTURE = pd.date_range(s["ds"].iloc[-1], periods=H + 1, freq=FREQ)[1:]
print("you are forecasting:", FUTURE[0].date(), "to", FUTURE[-1].date())

## 1. EDA and visualisation

*1 point.* Plot the series and say what you find. The plots are here to justify the decisions in
Section 2.

- a time plot, always;
- whichever of the seasonal, subseries and ACF plots your argument actually uses;
- the validation split on a plot: history, validation window, your forecast over it;
- one sentence per plot. A plot with no reading attached earns nothing.

In [ ]:
# TODO: plot your series and read it

**What the series shows.**

→

## 2. Decomposition and forecast

*3 points.* One for the decomposition, one for the forecast built out of it, one for the submission.

### 2.1 The decomposition

*1 point.* STL with `period=M`, components plotted, and the seasonally adjusted series computed from
it. A decomposition with the wrong period earns nothing here and nothing after it.

In [ ]:
# TODO: STL on your series. Plot observed, trend, seasonal, remainder, and compute the
# seasonally adjusted series (observed minus seasonal).

**Reading the components.**

→

### 2.2 The forecast, built from the parts

*1 point.* The recipe: forecast the **seasonally adjusted** series with a simple method, carry the
seasonal component forward, add them back. Judge candidates on the validation window — the last `H`
observations — against all four simple methods.

The split and the four benchmarks are given below; the candidates are yours.

In [ ]:
train, valid = y[:-H], y[-H:]

simple = benchmark_forecasts(train, H, M)
simple_scores = {name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
                 for name, fc in simple.items()}
BENCHMARK = min(simple_scores, key=lambda k: simple_scores[k]["rmse"])

print(pd.DataFrame(simple_scores).T.round(3).to_string())
print(f"\nbest of the four on the validation window: {BENCHMARK} "
      f"(MASE {simple_scores[BENCHMARK]['mase']:.3f})")

In [ ]:
# TODO: build your decomposition forecasts of the validation window and score them with mase().
#
#   fit = STL(train, period=M, seasonal=..., robust=...).fit()
#   adjusted = train - fit.seasonal
#   level = benchmark_forecasts(adjusted, H, M)["naive" | "drift" | "mean"]
#   season = np.array([fit.seasonal[-M + (i % M)] for i in range(H)])
#   candidate = level + season
#
# Keep a table of everything you tried: Section 3 asks for it.

**Your candidates, their validation scores, and the one you chose.**

→

### 2.3 The submission

*1 point.* Refit your chosen approach on the **whole** history `y` — the validation window goes back
into the data — and forecast `H` steps with an 80% interval.

The crude honest interval: the standard deviation of your one-step residuals, widened by
$\sqrt{k}$ at step $k$, times 1.28.

In [ ]:
# TODO: your final forecast, fitted on all of y.
YHAT = None          # np.array of length H
YHAT_LO_80 = None    # np.array of length H
YHAT_HI_80 = None    # np.array of length H

for name, value in [("YHAT", YHAT), ("YHAT_LO_80", YHAT_LO_80), ("YHAT_HI_80", YHAT_HI_80)]:
    if value is None or len(np.ravel(value)) != H:
        raise ValueError(f"{name} must be an array of length H = {H}")

## 3. Justification block

*3 points, and the part a human reads closely.* Roughly a paragraph each. Numbers from your own
cells, not adjectives.

**3.1 What the data shows** *(0.75)* — frequency, seasonal period, trend, level shifts, outliers,
each claim tied to a plot from Section 1.

→

**3.2 Why this decomposition and this level method** *(0.75)* — your choice of `seasonal` and
`robust`, and of the method that forecasts the adjusted series, tied to what you described in 3.1.
What did you try and drop?

→

**3.3 What the residuals say** *(0.75)* — ACF and a Ljung-Box test on your final model's residuals,
read: what structure is left, and what it damages — the point forecast or the interval.

→

In [ ]:
# TODO: residual diagnostics of your final model (ACF + Ljung-Box)

**3.4 What you expect** *(0.75)* — your validation MASE against all four simple methods, and what you
expect on the hidden horizon. If you expect a simple method to win, say which and why.

→

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- The submission file is written by the cell below, not edited by hand.
- No leakage: everything is fitted on `y` or on `train`, never on anything after them.
- Every number in your text comes from a cell.

## 5. Submit

Run both cells, then upload **three files** to the Week 4 activity on Moodle before 23:59:

- `SURNAME_lab04_forecast.csv` — written below;
- `SURNAME_lab04.pdf` — `File → Print → Save as PDF`;
- `SURNAME_lab04.ipynb` — `File → Download → Download .ipynb`.

In [ ]:
submission = pd.DataFrame({
    "student_id": MY_ID,
    "code": CODE,
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": np.ravel(YHAT).astype(float),
    "yhat_lo_80": np.ravel(YHAT_LO_80).astype(float),
    "yhat_hi_80": np.ravel(YHAT_HI_80).astype(float),
})

name = f"{SURNAME.strip().upper()}_lab04_forecast.csv"
submission.to_csv(name, index=False)
print(submission.to_string(index=False))

In [ ]:
# The same checks used when your file is read. If this cell passes, your file is fine.
check = pd.read_csv(name, parse_dates=["ds"])
assert len(check) == H, f"expected {H} rows, got {len(check)}"
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert (check["ds"].to_numpy() == FUTURE.to_numpy()).all(), "the dates are not the H dates asked for"
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all(), "no NaN allowed"
assert (check["yhat_lo_80"] <= check["yhat"]).all() and (check["yhat"] <= check["yhat_hi_80"]).all()
print("format OK -", name, "is ready to upload")

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"Not in Colab - {name} is saved next to this notebook.")